In [ ]:
import os, sys

if not os.path.isdir("Evaluation"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("Evaluation/ exists:", os.path.isdir("Evaluation"))


cwd: c:\Users\Offic\OneDrive\Desktop\Projects\5 - XAI Evaluation Paper
Evaluation/ exists: True


In [2]:
from src.evaluation import create_experiment
from src.robustness1 import run_robustness
from src.utils import Logger

os.makedirs("Evaluation/logs", exist_ok=True)
# logger = Logger("Evaluation/logs", filename="setup_log.txt")
# experiment_id = create_experiment("Evaluation", {"purpose": "fidelity run"

DATASETS = [
    ("healthcare", "pima_diabetes"),
    ("healthcare", "breast_cancer_wisconsin"),
    ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_NAMES = ["rf", "xgb"]

logger = Logger("Evaluation/logs", filename="ledger.log")
experiment_id = create_experiment(
    "Evaluation",
    {"phase": "robustness", "datasets": DATASETS, "models": MODEL_NAMES},
    logger,
)
print("experiment_id:", experiment_id)

Created experiment_id=exp_20260718T145336_44b13d9a, logged to Evaluation\run_ledger.csv
experiment_id: exp_20260718T145336_44b13d9a


In [3]:
# import os
# from src.robustness import run_robustness

results = {}
for domain, dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        run_dir = os.path.join("Explanations", domain, dataset_name, model_name)
        if not os.path.isdir(run_dir):
            print(f"SKIP {domain}/{dataset_name}/{model_name}: no Explanations/ artifacts yet")
            continue
        try:
            rows = run_robustness({
                "domain": domain,
                "dataset_name": dataset_name,
                "model_name": model_name,
                "experiment_id": experiment_id,
                "robustness_root": "Evaluation/Robustness",
            })
            results[(domain, dataset_name, model_name)] = len(rows)
        except Exception as e:
            print(f"FAILED {domain}/{dataset_name}/{model_name}: {e}")

print()
print(f"{len(results)}/{len(DATASETS) * len(MODEL_NAMES)} combos completed.")
results


------------------------------------------------------------
ROBUSTNESS: pima_diabetes x rf
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\rf\model\model.skops
Feature-group map for pima_diabetes: 8 original features (from 8 model-input columns).
Computed std for 8 numerical feature(s) from Datasets\healthcare\pima_diabetes\processed\data\X_train_prebalance.csv.
Wrote 1232 rows to Evaluation/Robustness\healthcare\pima_diabetes\rf\shap\perturbation_attributions.csv
shap: robustness computed for 154 instance(s).
Wrote 1232 rows to Evaluation/Robustness\healthcare\pima_diabetes\rf\lime\perturbation_attributions.csv
lime: robustness computed for 154 instance(s).
Appended 616 row(s) to Evaluation\metrics_long.csv

------------------------------------------------------------
DONE
------------------------------------------------------------
Robustness: 616 metrics_long rows written across ['shap', 'lime'].

-----------------

{('healthcare', 'pima_diabetes', 'rf'): 616,
 ('healthcare', 'pima_diabetes', 'xgb'): 616,
 ('healthcare', 'breast_cancer_wisconsin', 'rf'): 456,
 ('healthcare', 'breast_cancer_wisconsin', 'xgb'): 456,
 ('healthcare', 'heart_disease_uci', 'rf'): 736,
 ('healthcare', 'heart_disease_uci', 'xgb'): 736,
 ('finance', 'loan_default', 'rf'): 2000,
 ('finance', 'loan_default', 'xgb'): 2000,
 ('finance', 'financial_distress', 'rf'): 2936,
 ('finance', 'financial_distress', 'xgb'): 2936,
 ('finance', 'credit_card_fraud_2023', 'rf'): 2000,
 ('finance', 'credit_card_fraud_2023', 'xgb'): 2000}

In [4]:
import pandas as pd

def feature_volatility(domain, dataset_name, model_name, explainer):
    path = f"Evaluation/Robustness/{domain}/{dataset_name}/{model_name}/{explainer}/perturbation_attributions.csv"
    raw = pd.read_csv(path)
    raw["abs_diff"] = (raw["original_attribution"] - raw["perturbed_attribution"]).abs()
    return raw.groupby("feature")["abs_diff"].mean().sort_values(ascending=False)

feature_volatility("healthcare", "pima_diabetes", "rf", "shap")


feature
Glucose                     0.007472
BMI                         0.005441
Age                         0.005205
Insulin                     0.003723
SkinThickness               0.003094
DiabetesPedigreeFunction    0.002174
Pregnancies                 0.002014
BloodPressure               0.001054
Name: abs_diff, dtype: float64

In [5]:
shap_vol = feature_volatility("healthcare", "pima_diabetes", "rf", "shap").rename("shap")
lime_vol = feature_volatility("healthcare", "pima_diabetes", "rf", "lime").rename("lime")
pd.concat([shap_vol, lime_vol], axis=1)

,shap,lime
feature,,
Glucose,0.007472,0.008850
BMI,0.005441,0.002698
Age,0.005205,0.002548
Insulin,0.003723,0.011323
SkinThickness,0.003094,0.003374
DiabetesPedigreeFunction,0.002174,0.004274
Pregnancies,0.002014,0.002573
BloodPressure,0.001054,0.001688


In [6]:
raw = pd.read_csv("Evaluation/Robustness/healthcare/pima_diabetes/rf/shap/perturbation_attributions.csv")
raw.groupby("feature").apply(lambda g: g["original_attribution"].corr(g["perturbed_attribution"]), include_groups=False).sort_values()

feature
SkinThickness               0.940610
Pregnancies                 0.954292
BloodPressure               0.955259
Age                         0.955564
DiabetesPedigreeFunction    0.978875
Insulin                     0.981801
BMI                         0.987502
Glucose                     0.988423
dtype: float64

In [ ]:
import pandas as pd
import os

def discover_robustness_combos(robustness_root="Evaluation/Robustness"):
    combos = []
    if not os.path.isdir(robustness_root):
        return combos
    for domain in sorted(os.listdir(robustness_root)):
        d_path = os.path.join(robustness_root, domain)
        if not os.path.isdir(d_path): continue
        for dataset in sorted(os.listdir(d_path)):
            ds_path = os.path.join(d_path, dataset)
            if not os.path.isdir(ds_path): continue
            for model in sorted(os.listdir(ds_path)):
                m_path = os.path.join(ds_path, model)
                for explainer in sorted(os.listdir(m_path)):
                    f = os.path.join(m_path, explainer, "perturbation_attributions.csv")
                    if os.path.exists(f):
                        combos.append((domain, dataset, model, explainer, f))
    return combos

combos = discover_robustness_combos()
print(f"Found {len(combos)} combo(s) with raw robustness artifacts.\n")

agg_rows = []
for domain, dataset, model, explainer, path in combos:
    raw = pd.read_csv(path)
    raw["abs_diff"] = (raw["original_attribution"] - raw["perturbed_attribution"]).abs()
    agg_rows.append({
        "domain": domain, "dataset": dataset, "model": model, "explainer": explainer,
        "mean_abs_attribution_shift": raw["abs_diff"].mean(),
        "most_volatile_feature": raw.groupby("feature")["abs_diff"].mean().idxmax(),
    })

agg_df = pd.DataFrame(agg_rows).sort_values(["domain", "dataset", "model", "explainer"])
agg_df


Found 24 combo(s) with raw robustness artifacts.



,domain,dataset,model,explainer,mean_abs_attribution_shift,most_volatile_feature
0,finance,credit_card_fraud_2023,rf,lime,0.002136,V10
1,finance,credit_card_fraud_2023,rf,shap,0.001804,V4
2,finance,credit_card_fraud_2023,xgb,lime,0.004651,V8
3,finance,credit_card_fraud_2023,xgb,shap,0.104727,V8
4,finance,financial_distress,rf,lime,0.003500,x47
5,finance,financial_distress,rf,shap,0.001979,x16
6,finance,financial_distress,xgb,lime,0.008927,x52
7,finance,financial_distress,xgb,shap,0.045583,x52
8,finance,loan_default,rf,lime,0.000863,LoanTerm
9,finance,loan_default,rf,shap,0.005767,NumCreditLines


In [9]:
import pandas as pd
import os

def discover_robustness_combos(robustness_root="Evaluation/Robustness"):
    combos = []
    for domain in sorted(os.listdir(robustness_root)):
        d_path = os.path.join(robustness_root, domain)
        for dataset in sorted(os.listdir(d_path)):
            ds_path = os.path.join(d_path, dataset)
            for model in sorted(os.listdir(ds_path)):
                m_path = os.path.join(ds_path, model)
                for explainer in sorted(os.listdir(m_path)):
                    f = os.path.join(m_path, explainer, "perturbation_attributions.csv")
                    if os.path.exists(f):
                        combos.append((domain, dataset, model, explainer, f))
    return combos

def feature_volatility(path):
    raw = pd.read_csv(path)
    raw["abs_diff"] = (raw["original_attribution"] - raw["perturbed_attribution"]).abs()
    return raw.groupby("feature")["abs_diff"].mean()

combos = discover_robustness_combos()

grouped = {}
for domain, dataset, model, explainer, path in combos:
    grouped.setdefault((domain, dataset, model), {})[explainer] = feature_volatility(path)

rows = []
for (domain, dataset, model), by_explainer in grouped.items():
    if "shap" not in by_explainer or "lime" not in by_explainer:
        continue
    shap_vol, lime_vol = by_explainer["shap"], by_explainer["lime"]
    for feat in sorted(set(shap_vol.index) | set(lime_vol.index)):
        s, l = shap_vol.get(feat, 0.0), lime_vol.get(feat, 0.0)
        rows.append({
            "domain": domain, "dataset": dataset, "model": model, "feature": feat,
            "shap_volatility": s, "lime_volatility": l,
            "more_fragile": "SHAP" if s > l else ("LIME" if l > s else "tie"),
        })

comparison = pd.DataFrame(rows)
comparison

,domain,dataset,model,feature,shap_volatility,lime_volatility,more_fragile
0,finance,credit_card_fraud_2023,rf,Amount,0.000026,0.001308,LIME
1,finance,credit_card_fraud_2023,rf,V1,0.001821,0.001435,SHAP
2,finance,credit_card_fraud_2023,rf,V10,0.003807,0.004087,LIME
3,finance,credit_card_fraud_2023,rf,V11,0.002478,0.002632,LIME
4,finance,credit_card_fraud_2023,rf,V12,0.004344,0.002811,SHAP
...,...,...,...,...,...,...,...
355,healthcare,pima_diabetes,xgb,DiabetesPedigreeFunction,0.006764,0.002690,SHAP
356,healthcare,pima_diabetes,xgb,Glucose,0.034924,0.008866,SHAP
357,healthcare,pima_diabetes,xgb,Insulin,0.013964,0.007722,SHAP
358,healthcare,pima_diabetes,xgb,Pregnancies,0.005721,0.001976,SHAP


In [12]:
df = pd.read_csv("Evaluation/metrics_long.csv")

rob = df[df.metric_property == "Robustness"]
display(rob.groupby(["dataset","model","explainer"])["value"].mean())  # Spearman/Jaccard, already scale-free

C:\Users\Offic\AppData\Local\Temp\ipykernel_22440\1764127541.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Evaluation/metrics_long.csv")


dataset                  model  explainer
breast_cancer_wisconsin  rf     lime         0.948618
                                shap         0.929680
                         xgb    lime         0.930221
                                shap         0.938642
credit_card_fraud_2023   rf     lime         0.923584
                                shap         0.938526
                         xgb    lime         0.901689
                                shap         0.895818
financial_distress       rf     lime         0.675612
                                shap         0.736929
                         xgb    lime         0.627503
                                shap         0.662313
heart_disease_uci        rf     lime         0.962031
                                shap         0.974090
                         xgb    lime         0.956746
                                shap         0.920384
loan_default             rf     lime         0.977803
                                shap    